In [5]:
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [9]:
#prompt = f"""
#Please provide a solution to the following task:
#{task}
#"""

In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [6]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that parses an AWS S3 bucket URI (e.g., 's3://my-bucket/path/to/file.txt') and returns a dictionary with 'bucket' and 'key' as separate fields."}, {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to all objects in an S3 bucket named 'data-lake'."}, {'task': "Write a regex pattern that validates AWS EC2 security group IDs, which follow the format 'sg-' followed by exactly 17 hexadecimal characters (e.g., 'sg-0a1b2c3d4e5f6g7h8')."}]


In [7]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [8]:
print(open("dataset.json").read())

[
  {
    "task": "Write a Python function that parses an AWS S3 bucket URI (e.g., 's3://my-bucket/path/to/file.txt') and returns a dictionary with 'bucket' and 'key' as separate fields."
  },
  {
    "task": "Create a JSON object that represents an AWS IAM policy allowing read-only access to all objects in an S3 bucket named 'data-lake'."
  },
  {
    "task": "Write a regex pattern that validates AWS EC2 security group IDs, which follow the format 'sg-' followed by exactly 17 hexadecimal characters (e.g., 'sg-0a1b2c3d4e5f6g7h8')."
  }
]
